# 🛡️ Bank Shield AI - Fraud Detection MVP (Minimum Viable Product)

Bu çalışma, ham işlem verilerinden anlamlı özellikler türeterek bir sahtekarlık tespit modeli oluşturmayı amaçlar. 

### 🚀 Uygulanan Temel Adımlar:
1. **Veri Ön İşleme:** Tip dönüşümleri ve ID senkronizasyonu.
2. **Feature Engineering:** - *Zaman:* Gece/Gündüz, Hafta sonu, Yoğun saatler.
    - *Müşteri Davranışı:* Harcama limit oranı, harcama sapması (Z-Score), son 3 işlem ortalaması.
    - *Merchant Risk:* MCC kodlarına göre risk ağırlıklandırma.
3. **Model:** `LogisticRegression` (Sınıf ağırlıkları dengelenmiş - Balanced).
4. **Değerlendirme:** Recall (Geri Çağırma) odaklı analiz (Fraud vakalarını kaçırmamak için).

**Resampling Hakkında:** > "Veri setinde ciddi bir dengesizlik (Imbalance) olduğu için, eğitim sırasında sınıfları 1:20 oranında dengeleyerek modelin Fraud vakalarını öğrenmesi sağlanmıştır."

**Metric Seçimi:** > "Bu projede Recall (Duyarlılık) metriği önceliklendirilmiştir. Amacımız, sahtekarlık vakalarını %94 oranında yakalayabilmektir."


In [45]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

In [46]:
df = pd.read_parquet("../data/processed/df_combined.parquet")

In [47]:
print(df.shape)
df.head()

(13305915, 46)


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,year_pin_last_changed,card_on_dark_web,mcc_description,is_fraud,year,month,day,hour,day_of_week,is_weekend
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,2008,No,Miscellaneous Food Stores,0,2010,1,1,0,Friday,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,2015,No,Department Stores,0,2010,1,1,0,Friday,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,2008,No,Money Transfer,0,2010,1,1,0,Friday,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,2006,No,Money Transfer,0,2010,1,1,0,Friday,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,2014,No,Drinking Places (Alcoholic Beverages),0,2010,1,1,0,Friday,0


In [48]:
df = df.copy()

# varsa gereksiz index kolonu sil
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

rename_map = {}

if "id" in df.columns and "transaction_id" not in df.columns:
    rename_map["id"] = "transaction_id"

if "date" in df.columns and "transaction_datetime" not in df.columns:
    rename_map["date"] = "transaction_datetime"

if "zip" in df.columns and "merchant_zip" not in df.columns:
    rename_map["zip"] = "merchant_zip"

df = df.rename(columns=rename_map)

print(df.columns.tolist())
print(df.shape)

['transaction_id', 'transaction_datetime', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'merchant_zip', 'mcc', 'errors', 'is_return', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'client_id_card', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'mcc_description', 'is_fraud', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']
(13305915, 46)


### 🛠️ Veri Tipleme ve Format Düzenleme

Modelin sağlıklı çalışabilmesi ve bellek yönetimini optimize etmek adına veri tiplerini düzenliyoruz:

* **Kimlik Bilgileri (IDs):** `transaction_id`, `client_id` ve `card_id` gibi kolonlar sayısal büyüklük ifade etmediği için **String (Object)** tipine dönüştürülmüştür.
* **Tarih-Saat:** Zaman bazlı analizler ve kronolojik sıralama için ilgili sütun **Datetime** formatına çevrilmiştir.
* **Finansal Kolonlar:** Para birimi sembolleri veya özel karakterler içeren `credit_limit`, `yearly_income` gibi sütunlar temizlenerek **Numeric (Float)** formata getirilmiştir.
* **Target Variable (is_fraud):** Hedef değişken boş değerlerden arındırılmış ve modelin ikili sınıflandırma (binary classification) yapabilmesi için **Integer** tipine sabitlenmiştir.

> **Not:** Target değişkenindeki dağılımın aşırı dengesiz (imbalance) olduğu gözlemlenmiştir. Bu durum ilerleyen aşamalarda resampling stratejisini zorunlu kılmaktadır.

In [49]:
# id benzeri kolonlar string olsun
df["transaction_id"] = df["transaction_id"].astype(str)
df["client_id"] = df["client_id"].astype(str)
df["card_id"] = df["card_id"].astype(str)

# datetime
df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"], errors="coerce")

# sayısal kolonlar
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["is_fraud"] = pd.to_numeric(df["is_fraud"], errors="coerce").fillna(0).astype(int)

money_cols = ["credit_limit", "yearly_income", "per_capita_income", "total_debt"]

for col in money_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[^0-9\.\-]", "", regex=True)
            .replace("", np.nan)
            .pipe(pd.to_numeric, errors="coerce")
        )

print(df.dtypes.head(15))
print(df["is_fraud"].value_counts(dropna=False))

transaction_id                     str
transaction_datetime    datetime64[us]
client_id                          str
card_id                            str
amount                         float64
use_chip                           str
merchant_id                      int64
merchant_city                      str
merchant_state                     str
merchant_zip                       str
mcc                                str
errors                             str
is_return                         bool
current_age                      int64
retirement_age                   int64
dtype: object
is_fraud
0    13292583
1       13332
Name: count, dtype: int64


### 🕒 Katman 1: Zaman Bazlı Özellikler (Time-based Features)
Ham tarih verisinden modelin gün içi döngüleri ve hafta sonu davranışlarını anlayabilmesi için 'is_night', 'is_weekend' ve 'is_peak_hour' gibi bayraklar (flags) türetilmiştir.

In [50]:
#Bu adımda yaptığımız şey:

#tek bir tarih kolonunu, modelin anlayacağı birden fazla anlamlı kolona çevirmek
df["hour"] = df["transaction_datetime"].dt.hour
df["day"] = df["transaction_datetime"].dt.day
df["month"] = df["transaction_datetime"].dt.month
df["year"] = df["transaction_datetime"].dt.year
df["day_of_week_num"] = df["transaction_datetime"].dt.dayofweek

if "day_of_week" not in df.columns:
    df["day_of_week"] = df["transaction_datetime"].dt.day_name()

df["is_weekend"] = (df["day_of_week_num"] >= 5).astype(int)
df["is_night"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)
df["is_peak_hour"] = df["hour"].isin([12, 13, 18, 19, 20]).astype(int)

df[["transaction_datetime", "hour", "day", "month", "day_of_week", "is_weekend", "is_night"]].head()

,transaction_datetime,hour,day,month,day_of_week,is_weekend,is_night
0,2010-01-01 00:01:00,0,1,1,Friday,0,1
1,2010-01-01 00:02:00,0,1,1,Friday,0,1
2,2010-01-01 00:02:00,0,1,1,Friday,0,1
3,2010-01-01 00:05:00,0,1,1,Friday,0,1
4,2010-01-01 00:06:00,0,1,1,Friday,0,1


### 💳 Katman 2: İşlem ve Limit Analizi
İşlem tutarları üzerinde normalizasyon (Log Transform) ve standartlaştırma (Z-Score) uygulanmıştır. Ayrıca harcamanın kredi limitine oranı hesaplanarak, limit zorlayan şüpheli işlemler hedeflenmiştir.

In [51]:
#Burada işlem tutarından türeyen feature’ları üretiyoruz.
df["log_amount"] = np.log1p(df["amount"].clip(lower=0))

amount_mean = df["amount"].mean()
amount_std = df["amount"].std()

if pd.notna(amount_std) and amount_std != 0:
    df["amount_zscore"] = (df["amount"] - amount_mean) / amount_std
else:
    df["amount_zscore"] = 0

df["high_amount"] = (df["amount"] > df["amount"].quantile(0.95)).astype(int)

if "credit_limit" in df.columns:
    df["amount_to_limit_ratio"] = np.where(
        df["credit_limit"].fillna(0) > 0,
        df["amount"] / df["credit_limit"],
        np.nan
    )
else:
    df["amount_to_limit_ratio"] = np.nan

df[["amount", "log_amount", "amount_zscore", "high_amount", "amount_to_limit_ratio"]].head()

,amount,log_amount,amount_zscore,high_amount,amount_to_limit_ratio
0,77.00,4.356709,0.316447,0,1.400000
1,14.57,2.745346,-0.511013,0,0.001601
2,80.00,4.394449,0.356210,0,0.005405
3,200.00,5.303305,1.946715,1,0.005314
4,46.41,3.858833,-0.088999,0,0.002428




`use_chip` sütunundaki kategorik veriler, işlemin fiziksel mi yoksa dijital ortamda mı gerçekleştiğini anlamak adına ayrıştırılmıştır:

* **is_online:** İşlemin internet üzerinden yapılıp yapılmadığını belirtir. (Online işlemler genellikle daha yüksek risk taşır.)
* **is_chip_used:** Kartın çip kullanılarak (fiziksel POS) işlem görüp görmediğini gösterir.
* **is_swipe:** Kartın manyetik şerit kaydırma yöntemiyle kullanılıp kullanılmadığını ifade eder. (Legacy/eski yöntemler güvenlik riski oluşturabilir.)

> **Amaç:** Kategorik bir değişkeni modelin ağırlıklandırabileceği sayısal (binary) özelliklere dönüştürerek işlem tipinin dolandırıcılık üzerindeki etkisini ölçmek.

In [52]:

if "use_chip" in df.columns:
    use_chip_str = df["use_chip"].astype(str).str.lower()
    df["is_online"] = use_chip_str.str.contains("online", na=False).astype(int)
    df["is_chip_used"] = use_chip_str.str.contains("chip", na=False).astype(int)
    df["is_swipe"] = use_chip_str.str.contains("swipe", na=False).astype(int)
else:
    df["is_online"] = 0
    df["is_chip_used"] = 0
    df["is_swipe"] = 0

df[["use_chip", "is_online", "is_chip_used", "is_swipe"]].head()

,use_chip,is_online,is_chip_used,is_swipe
0,Swipe Transaction,0,0,1
1,Swipe Transaction,0,0,1
2,Swipe Transaction,0,0,1
3,Swipe Transaction,0,0,1
4,Swipe Transaction,0,0,1


### 👤 Katman 3: Müşteri Geçmişi ve Davranışsal Trendler

Statik verilerin ötesine geçerek, her müşterinin geçmiş işlem alışkanlıklarına göre dinamik özellikler (Behavioral Features) türetiyoruz:

* **Harcama Profili:** Müşterinin genel ortalaması (`user_mean_amount`) ve mevcut işlemin bu ortalamadan sapması (`amount_deviation`) hesaplanmıştır.
* **Hız Kontrolü (Velocity Checks):** İki işlem arasındaki süre (`time_diff`) hesaplanarak; 60 saniyeden kısa (`fast_tx`) ve 10 saniyeden kısa (`very_fast_tx`) sürelerde peş peşe yapılan şüpheli işlemler işaretlenmiştir.
* **Hareketli İstatistikler (Rolling Stats):** Son 3 işlem özelinde hareketli ortalama ve standart sapma hesaplanarak, ani harcama değişiklikleri modelin algısına sunulmuştur.

> **Önem:** Sahtekarlık tespiti, genellikle müşterinin "normal" davranışından sapmasıyla (anomali) yakalanır. Bu özellikler, modelin bu değişimleri fark etmesini sağlar.

In [53]:

df = df.sort_values(["client_id", "transaction_datetime"]).reset_index(drop=True)

df["user_tx_count"] = df.groupby("client_id")["transaction_id"].transform("count")
df["user_mean_amount"] = df.groupby("client_id")["amount"].transform("mean")
df["user_std_amount"] = df.groupby("client_id")["amount"].transform("std")
df["amount_deviation"] = df["amount"] - df["user_mean_amount"]

df["time_diff"] = (
    df.groupby("client_id")["transaction_datetime"]
      .diff()
      .dt.total_seconds()
)

df["fast_tx"] = (df["time_diff"] < 60).astype(int)
df["very_fast_tx"] = (df["time_diff"] < 10).astype(int)

df["rolling_mean_3"] = (
    df.groupby("client_id")["amount"]
      .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

df["rolling_std_3"] = (
    df.groupby("client_id")["amount"]
      .transform(lambda x: x.rolling(3, min_periods=1).std())
)

df["rolling_std_3"] = df["rolling_std_3"].fillna(0)
df["rolling_amount_deviation"] = df["amount"] - df["rolling_mean_3"]

df[[
    "client_id", "amount", "user_mean_amount", "amount_deviation",
    "time_diff", "fast_tx", "very_fast_tx",
    "rolling_mean_3", "rolling_std_3", "rolling_amount_deviation"
]].head(10)

,client_id,amount,user_mean_amount,amount_deviation,time_diff,fast_tx,very_fast_tx,rolling_mean_3,rolling_std_3,rolling_amount_deviation
0,0,33.96,61.03319,-27.07319,NaN,0,0,33.960000,0.000000,0.000000
1,0,7.78,61.03319,-53.25319,23340.0,0,0,20.870000,18.512056,-13.090000
2,0,65.86,61.03319,4.82681,9240.0,0,0,35.866667,29.086907,29.993333
3,0,55.85,61.03319,-5.18319,53700.0,0,0,43.163333,31.048917,12.686667
4,0,1.37,61.03319,-59.66319,95760.0,0,0,41.026667,34.706461,-39.656667
5,0,39.91,61.03319,-21.12319,22080.0,0,0,32.376667,28.010372,7.533333
6,0,167.62,61.03319,106.58681,39420.0,0,0,69.633333,87.019383,97.986667
7,0,28.68,61.03319,-32.35319,15240.0,0,0,78.736667,77.179748,-50.056667
8,0,30.32,61.03319,-30.71319,960.0,0,0,75.540000,79.747835,-45.220000
9,0,2.34,61.03319,-58.69319,4800.0,0,0,20.446667,15.702259,-18.106667


Müşterinin geçmişteki harcama alışkanlıkları (Rolling Mean/Std) ve işlemler arası süre (Time Diff) hesaplanarak; "hızlı işlem yapma" (fast_tx) gibi anomaliler yakalanmıştır.

Müşteri Kapsamı (Unique Clients)

Davranışsal analizlerimizin kaç farklı kişi üzerinden genellendiğini anlamak için veri setindeki toplam tekil müşteri sayısını doğruluyoruz. Bu, modelin farklı harcama profillerini tanıma kapasitesini gösterir.

In [54]:
df["client_id"].nunique()

1219

### 🏪 Üye İşyeri Risk Analizi (Merchant Category Risk)

Bazı sektörlerin (MCC - Merchant Category Code) doğası gereği dolandırıcılık vakalarına daha açık olduğu bilinmektedir. Bu hücrede, her bir kategori için risk ağırlıklandırması yapıyoruz:

* **Eşik Değer (Threshold):** İstatistiksel olarak anlamlı olması için en az 100 işlem görmüş kategoriler baz alınmıştır.
* **Riskli Kategoriler (Merchant Risk Flag):** Bir kategorideki fraud oranı, genel ortalamanın **2 katından fazlaysa**, bu kategori "Yüksek Riskli" olarak işaretlenmiştir.
* **Global Karşılaştırma:** Kategorik riskler, tüm veri setindeki `global_fraud_rate` ile kıyaslanarak dinamik bir bayrak (flag) sistemine dönüştürülmüştür.

> **Strateji:** Bu yöntemle model, işlemin yapıldığı yerin (örneğin; kuyumcu veya online bahis gibi yüksek riskli alanlar) tehlike seviyesini tek bir sayısal girdi üzerinden öğrenir.

In [55]:
if "mcc" in df.columns:
    mcc_stats = (
        df.groupby("mcc")["is_fraud"]
          .agg(["mean", "count"])
          .reset_index()
          .rename(columns={"mean": "mcc_fraud_rate", "count": "mcc_total_count"})
    )

    global_fraud_rate = df["is_fraud"].mean()

    mcc_stats["merchant_risk_flag"] = np.where(
        (mcc_stats["mcc_total_count"] >= 100) &
        (mcc_stats["mcc_fraud_rate"] > global_fraud_rate * 2),
        1,
        0
    )

    df = df.merge(
        mcc_stats[["mcc", "merchant_risk_flag"]],
        on="mcc",
        how="left"
    )

    df["merchant_risk_flag"] = df["merchant_risk_flag"].fillna(0).astype(int)
else:
    df["merchant_risk_flag"] = 0

df[["mcc", "merchant_risk_flag"]].head()

,mcc,merchant_risk_flag
0,5942,0
1,5812,0
2,5813,0
3,5211,0
4,5499,0


### 🕵️ En Riskli Sektörlerin Tespiti

Hesapladığımız istatistikler sonucunda sahtekarlık oranının (Fraud Rate) en yüksek olduğu ilk 10 MCC (Üye İşyeri Kategori Kodu) listelenmiştir. 

* **Analiz:** Bu tablo, modelin hangi sektörleri "sabıkalı" olarak göreceğini doğrulamamızı sağlar.
* **Kritik Gözlem:** Toplam işlem sayısı (`mcc_total_count`) ile fraud oranı arasındaki denge, modelin gürültülü veriden (tesadüfi fraudlar) ziyade kronik riskli alanlara odaklandığını gösterir.

In [56]:
mcc_stats.sort_values("mcc_fraud_rate", ascending=False).head(10)

,mcc,mcc_fraud_rate,mcc_total_count,merchant_risk_flag
38,4411,0.385514,428,1
68,5733,0.238245,319,1
4,3006,0.082621,351,1
46,5045,0.073040,2793,1
12,3144,0.068862,334,1
67,5732,0.057453,6997,1
3,3005,0.056266,391,1
7,3009,0.053922,408,1
47,5094,0.046718,5180,1
5,3007,0.044619,381,1


In [57]:
#Numerik boş değerleri temizleme
num_cols_all = df.select_dtypes(include=[np.number]).columns
df[num_cols_all] = df[num_cols_all].fillna(0)

print(df.isna().sum().sort_values(ascending=False).head(20))

errors              13094522
merchant_zip         1652706
merchant_state       1563700
hour                       0
log_amount                 0
is_peak_hour               0
is_night                   0
day_of_week_num            0
is_weekend                 0
day_of_week                0
transaction_id             0
high_amount                0
day                        0
month                      0
year                       0
is_fraud                   0
mcc_description            0
card_on_dark_web           0
amount_zscore              0
is_online                  0
dtype: int64


In [58]:
df["errors"] = df["errors"].fillna("no_error")

In [59]:
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

### 🎯 MVP Model Tablosunun Oluşturulması (Feature Selection)

Türetilen tüm özellikler ve ham veriler arasından, modelin eğitilmesinde kullanılacak en anlamlı değişkenler (Features) seçilerek nihai model tablosu oluşturulmuştur:

* **Özellik Grupları:** Finansal metrikler, zaman döngüleri, işlem yöntemleri, müşteri davranış trendleri ve üye işyeri risk flag'leri bir araya getirilmiştir.
* **Veri Filtreleme:** Sadece seçilen `feature_cols` ve hedef değişken olan `is_fraud` sütunları alınarak bellek kullanımı optimize edilmiştir.
* **Son Kontrol:** Veri setinin nihai boyutları (shape) ve sınıfların dağılımı (is_fraud counts) kontrol edilerek modelleme aşamasına geçiş için onay verilmiştir.

> **Strateji:** Gereksiz veya sızıntı (leakage) yaratabilecek sütunlar dışarıda bırakılarak, modelin genelleme yeteneği artırılmıştır.

In [60]:
feature_cols = [
    "amount",
    "log_amount",
    "amount_zscore",
    "high_amount",
    "amount_to_limit_ratio",
    "hour",
    "day",
    "month",
    "year",
    "day_of_week",
    "day_of_week_num",
    "is_weekend",
    "is_night",
    "is_peak_hour",
    "use_chip",
    "is_online",
    "is_chip_used",
    "is_swipe",
    "merchant_city",
    "merchant_state",
    "mcc",
    "merchant_risk_flag",
    "is_return",
    "current_age",
    "gender",
    "yearly_income",
    "per_capita_income",
    "total_debt",
    "credit_score",
    "user_tx_count",
    "user_mean_amount",
    "user_std_amount",
    "amount_deviation",
    "time_diff",
    "fast_tx",
    "very_fast_tx",
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_amount_deviation"
]

feature_cols = [c for c in feature_cols if c in df.columns]

df_model = df[feature_cols + ["is_fraud"]].copy()

print("Model tablosu shape:", df_model.shape)
print(df_model["is_fraud"].value_counts(dropna=False))

Model tablosu shape: (13305915, 40)
is_fraud
0    13292583
1       13332
Name: count, dtype: int64


### 🧠 Model Seçimi ve Dengesiz Veri Stratejisi
Veri setinde Fraud oranı yaklaşık **%0.1**'dir. Bu ciddi dengesizliği (Imbalance) yönetmek için:
1. Eğitim setinde **Undersampling** yapılarak oran 1:20'ye çekilmiştir.
2. Modelde `class_weight='balanced'` parametresi kullanılarak Fraud vakalarına daha fazla ağırlık verilmiştir.
3. Başarı kriteri olarak `Accuracy` yerine, Fraud yakalama gücünü gösteren `Recall` ve `Precision-Recall AUC` baz alınmıştır.

In [61]:
#MVP samlpe üret , yani tüm fraud kayıtlarını koru.
fraud_df = df_model[df_model["is_fraud"] == 1].copy()
nonfraud_df = df_model[df_model["is_fraud"] == 0].copy()

print("Fraud satır sayısı:", len(fraud_df))
print("Non-fraud satır sayısı:", len(nonfraud_df))

# Yaklaşık 1:20 oranı -> fraud oranı yaklaşık %5
desired_nonfraud = min(len(nonfraud_df), len(fraud_df) * 20)

nonfraud_sample = nonfraud_df.sample(n=desired_nonfraud, random_state=42)

df_mvp = pd.concat([fraud_df, nonfraud_sample], axis=0)
df_mvp = df_mvp.sample(frac=1, random_state=42).reset_index(drop=True)

print("MVP sample shape:", df_mvp.shape)
print(df_mvp["is_fraud"].value_counts())
print(df_mvp["is_fraud"].value_counts(normalize=True))

Fraud satır sayısı: 13332
Non-fraud satır sayısı: 13292583
MVP sample shape: (279972, 40)
is_fraud
0    266640
1     13332
Name: count, dtype: int64
is_fraud
0    0.952381
1    0.047619
Name: proportion, dtype: float64


In [62]:
#Train / Test split
X = df_mvp.drop(columns=["is_fraud"])
y = df_mvp["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud oranı:", y_train.mean())
print("Test fraud oranı:", y_test.mean())

Train shape: (223977, 39)
Test shape: (55995, 39)
Train fraud oranı: 0.047620961080825266
Test fraud oranı: 0.047611393874453074


## Sadece TRAIN ' de RESAMPLING 

In [63]:
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values

train_fraud = train_df[train_df["is_fraud"] == 1].copy()
train_nonfraud = train_df[train_df["is_fraud"] == 0].copy()

desired_train_nonfraud = min(len(train_nonfraud), len(train_fraud) * 20)

train_nonfraud_sample = train_nonfraud.sample(
    n=desired_train_nonfraud,
    random_state=42
)

train_balanced = pd.concat([train_fraud, train_nonfraud_sample], axis=0)
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = train_balanced.drop(columns=["is_fraud"])
y_train_balanced = train_balanced["is_fraud"]

print("Balanced train shape:", X_train_balanced.shape)
print(y_train_balanced.value_counts())
print(y_train_balanced.value_counts(normalize=True))

Balanced train shape: (223977, 39)
is_fraud
0    213311
1     10666
Name: count, dtype: int64
is_fraud
0    0.952379
1    0.047621
Name: proportion, dtype: float64


### ⚙️ Veri Ön İşleme Hattının (Pipeline) Hazırlanması

Modelin ham veriyi işleyebilmesi için otomatik bir boru hattı (Pipeline) kurguluyoruz. Bu aşamada ilk adım olarak sütunları türlerine göre ayrıştırıyoruz:

* **Sayısal Sütunlar (Numerical):** `amount`, `age`, `rolling_mean` gibi matematiksel değerler. Bunlar için ölçeklendirme (Scaling) gerekecektir.
* **Kategorik Sütunlar (Categorical):** `gender`, `merchant_city`, `use_chip` gibi metinsel değerler. Bunların modelin anlayacağı sayısal vektörlere (One-Hot Encoding) dönüştürülmesi gerekir.

> **Neden Pipeline?** Pipeline kullanımı, veri sızıntısını (Data Leakage) önler ve modelin yeni gelen veriler üzerinde her zaman aynı ön işleme adımlarını (Preprocessing) hatasız bir şekilde uygulamasını sağlar.

In [64]:
# Preprocessing pipeline hazırlamak
num_cols = X_train_balanced.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train_balanced.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerik kolon sayısı:", len(num_cols))
print("Kategorik kolon sayısı:", len(cat_cols))
print("Kategorik kolonlar:", cat_cols)

Numerik kolon sayısı: 32
Kategorik kolon sayısı: 7
Kategorik kolonlar: ['day_of_week', 'use_chip', 'merchant_city', 'merchant_state', 'mcc', 'is_return', 'gender']


In [65]:
# 1. Kolonları kesin olarak ayırıyoruz
cat_cols = ['day_of_week', 'use_chip', 'merchant_city', 'merchant_state', 'mcc', 'is_return', 'gender']

# X_train_balanced içindeki diğer tüm kolonları sayısal kabul et (is_fraud zaten y_train içinde)
num_cols = [col for col in X_train_balanced.columns if col not in cat_cols]

print(f"Sayısal kolonlar ({len(num_cols)} adet): {num_cols}")
print(f"Kategorik kolonlar ({len(cat_cols)} adet): {cat_cols}")

Sayısal kolonlar (32 adet): ['amount', 'log_amount', 'amount_zscore', 'high_amount', 'amount_to_limit_ratio', 'hour', 'day', 'month', 'year', 'day_of_week_num', 'is_weekend', 'is_night', 'is_peak_hour', 'is_online', 'is_chip_used', 'is_swipe', 'merchant_risk_flag', 'current_age', 'yearly_income', 'per_capita_income', 'total_debt', 'credit_score', 'user_tx_count', 'user_mean_amount', 'user_std_amount', 'amount_deviation', 'time_diff', 'fast_tx', 'very_fast_tx', 'rolling_mean_3', 'rolling_std_3', 'rolling_amount_deviation']
Kategorik kolonlar (7 adet): ['day_of_week', 'use_chip', 'merchant_city', 'merchant_state', 'mcc', 'is_return', 'gender']


In [66]:
#sayısal ve kategorik veriyi modele uygun formata hazırlamak
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # Burada eksik değerleri en çok tekrar edenle doldurur
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)) # sparse_output=False hatayı önlemeye yardımcı olur
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [67]:
# 1. Kolonları ayır
cat_cols = ['day_of_week', 'use_chip', 'merchant_city', 'merchant_state', 'mcc', 'is_return', 'gender']
num_cols = [c for c in X_train_balanced.columns if c not in cat_cols]

# 2. Pipeline'ı tek seferde kur
model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])),
    ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", solver='sag', random_state=42)) 
])

# 3. Eğit
model.fit(X_train_balanced.astype({c: str for c in cat_cols}), y_train_balanced)
print("✅ MVP Model Eğitildi!")

✅ MVP Model Eğitildi!


/Users/sema/GitHub/bank-shield-ai/venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
# Model eğitiminden hemen önce bunu yapıştır
X_train_balanced = X_train_balanced.convert_dtypes() # Karmaşık tipleri en uygun tipe çevirir
X_train_balanced[cat_cols] = X_train_balanced[cat_cols].astype(str) # Kategorikleri metne mühürle

: 

In [ ]:
#MVP modelini eğitelim
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train_balanced, y_train_balanced)
print("Model eğitildi.")

In [ ]:
#Modeli Değerlendirme
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nPrecision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Average Precision:", average_precision_score(y_test, y_proba))

#buradaki amaç modelimiz gerçekten fraud yakalıyor mu ? 


Confusion Matrix:
[[51571  1758]
 [  158  2508]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     53329
           1       0.59      0.94      0.72      2666

    accuracy                           0.97     55995
   macro avg       0.79      0.95      0.85     55995
weighted avg       0.98      0.97      0.97     55995


Precision: 0.5879043600562588
Recall: 0.940735183795949
F1: 0.723600692440854
ROC AUC: 0.9889760735482865
Average Precision: 0.8978791612358744


## 🏁 MVP Model Değerlendirmesi
Modelimiz test setinde **%94 Recall** başarısına ulaşmıştır. 
- **Anlamı:** Gerçekleşen 100 sahtekarlık işleminin 94 tanesi sistem tarafından başarıyla tespit edilebilmektedir. 
- **Gelişim Alanı:** Precision değerini artırmak için bir sonraki aşamada XGBoost veya Random Forest gibi daha kompleks algoritmalar denenecektir.

### 🚀 Modelin Kaydedilmesi ve Test Edilmesi (Inference)

Eğitilen MVP modeli, projenin diğer aşamalarında ve ekip üyeleri tarafından kullanılabilmesi için `joblib` formatında dışa aktarılmıştır. Bu hücrede modelin tekrar yüklenebilirliği ve tahmin yeteneği doğrulanmaktadır:

* **Persistence:** `fraud_mvp_model.joblib` dosyası yüklenerek modelin "hafızası" geri çağrılmıştır.
* **Prediction:** Test setinden alınan ilk 10 örnek üzerinde tahminleme yapılmıştır.
* **Probability:** Modelin sadece "Fraud" veya "Değil" demesiyle yetinmeyip, her işlem için atadığı **olasılık skorunu** (`fraud_probability`) da çıktıya ekleyerek karar verme sürecini şeffaflaştırıyoruz.

> **Sonuç:** Bu tablo, modelin canlı veriye (Inference) hazır olduğunu ve boru hattının (Pipeline) yeni verileri sorunsuz işlediğini kanıtlamaktadır.

**Hücre 1: Kayıt (Modeli ve Veriyi Arşive Gönderme)**

In [ ]:
import os
import joblib

# 1. Klasörleri hazırla (Yoksa oluşturur)
os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# 2. Ortak dosya yollarını değişkenlere ata
FINAL_MODEL_PATH = "../models/fraud_mvp_model.joblib"
FINAL_DATA_PATH = "../data/processed/transaction_features_mvp_sample.parquet"

# 3. Kayıt işlemleri
joblib.dump(model, FINAL_MODEL_PATH)
df_mvp.to_parquet(FINAL_DATA_PATH, index=False)

print(f"✅ Model ve MVP veri seti başarıyla arşivlendi.")

✅ Model ve MVP veri seti başarıyla arşivlendi.


**Hücre 2: Test Edilmesi- 🚀 Modelin Yeniden Kullanımı ve Tahmin Testi**
Aşağıdaki kod, kaydedilen modelin ekip üyeleri tarafından nasıl yükleneceğini ve tahmin (inference) yapılacağını simüle eder.

In [ ]:
# 4. Kaydedilen yolu kullanarak modeli geri yükle
loaded_model = joblib.load(FINAL_MODEL_PATH) 

# 5. Örnek tahminleri al (İlk 10 satır)
sample_data = X_test.head(10).copy()
sample_preds = loaded_model.predict(sample_data)
sample_proba = loaded_model.predict_proba(sample_data)[:, 1]

# 6. Sonuçları bir tabloda göster
sample_data["predicted_fraud"] = sample_preds
sample_data["fraud_probability"] = sample_proba

sample_data

AttributeError: 'LogisticRegression' object has no attribute 'coef_'